In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

load_dotenv(find_dotenv(usecwd=True))  

url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)
print("tersambung ke:", os.getenv("DB_NAME"))

tersambung ke: bca_credit_risk


In [2]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

rfm = pd.read_sql("SELECT * FROM account_rfm", engine)
print(rfm.shape)
print(rfm.describe().round(1))
print("tidak aktif di 1998:", (rfm["freq_12m"] == 0).sum())

(4500, 5)
       account_id  recency_days  freq_12m  cash_in_12m  avg_bal_12m
count      4500.0        4500.0    4500.0       4500.0       4492.0
mean       2786.1           2.6      71.6     213547.7      38964.4
std        2313.8          30.6      18.5     180981.4      19674.5
min           1.0           0.0       0.0          0.0     -27797.0
25%        1182.8           0.0      61.0      72354.0      22820.2
50%        2368.0           0.0      70.0     166550.0      35286.5
75%        3552.2           0.0      82.0     286822.2      52121.5
max       11382.0         858.0     155.0     978229.0     106380.0
tidak aktif di 1998: 8


In [3]:
aktif = rfm[rfm["freq_12m"] > 0].copy()          # rekening tidak aktif dipisah, bukan di-cluster
fitur_rfm = ["recency_days", "freq_12m", "cash_in_12m", "avg_bal_12m"]

Z = aktif[fitur_rfm].copy()
Z["avg_bal_12m"] = Z["avg_bal_12m"].clip(lower=0)
Z = np.log1p(Z)
Z = StandardScaler().fit_transform(Z)

In [4]:
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    print(k, round(silhouette_score(Z, km.labels_), 3))

2 0.754
3 0.431
4 0.437
5 0.439
6 0.337
7 0.331


In [5]:
K = 4
aktif["segmen"] = KMeans(n_clusters=K, n_init=10, random_state=42).fit_predict(Z)

profil = aktif.groupby("segmen")[fitur_rfm].median().round(0)
profil["n_rekening"] = aktif.groupby("segmen").size()
profil

,recency_days,freq_12m,cash_in_12m,avg_bal_12m,n_rekening
segmen,,,,,
0,0.0,78.0,280768.0,49391.0,2337
1,0.0,63.0,71794.0,22650.0,1976
2,27.0,23.0,33800.0,26534.0,164
3,0.0,42.0,110994.0,-5553.0,15


In [7]:
km2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Z)
print(pd.Series(km2.labels_).value_counts())

0    4322
1     170
Name: count, dtype: int64


In [6]:
nama = {0: "Aktif bernilai tinggi", 1: "Aktif bernilai rendah", 2: "Mulai pasif", 3: "Saldo minus"}
aktif["segmen_nama"] = aktif["segmen"].map(nama)

tidak = rfm[rfm["freq_12m"] == 0][["account_id"]].assign(segmen=-1, segmen_nama="Tidak aktif 1998")
seg = pd.concat([aktif[["account_id", "segmen", "segmen_nama"]], tidak])

seg.to_sql("account_segment", engine, if_exists="replace", index=False)
print(len(seg))
print(seg["segmen_nama"].value_counts())

4500
segmen_nama
Aktif bernilai tinggi    2337
Aktif bernilai rendah    1976
Mulai pasif               164
Saldo minus                15
Tidak aktif 1998            8
Name: count, dtype: int64


In [8]:
for t in ["disp", "card"]:
    print(t, pd.read_sql(f"SELECT * FROM {t} LIMIT 2", engine).columns.tolist())

disp ['disp_id', 'client_id', 'account_id', 'type']
card ['card_id', 'disp_id', 'type', 'issued']
